In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import ccxt
import talib
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

st.set_page_config(page_title="BTC Price Predictor", layout="centered")

# ---------- UTILITY FUNCTIONS ----------
@st.cache_data
def fetch_data(symbol='BTC/USDT', timeframe='1h', limit=1000):
    exchange = ccxt.binance({'enableRateLimit': True})
    data = exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df

def compute_indicators(df):
    df['MA_10'] = df['close'].rolling(window=10).mean()
    df['MA_50'] = df['close'].rolling(window=50).mean()
    df['EMA_10'] = df['close'].ewm(span=10).mean()
    df['EMA_50'] = df['close'].ewm(span=50).mean()
    df['RSI'] = talib.RSI(df['close'], timeperiod=14)
    df['MACD'], df['MACD_signal'], _ = talib.MACD(df['close'], 12, 26, 9)
    df['Upper_BB'], df['Middle_BB'], df['Lower_BB'] = talib.BBANDS(df['close'], 20)
    df['HV'] = df['close'].pct_change().rolling(20).std()
    df['ATR'] = talib.ATR(df['high'], df['low'], df['close'], timeperiod=14)
    return df.dropna()

def prepare_data(df):
    features = df[['open', 'high', 'low', 'close', 'volume', 'MA_10', 'MA_50',
                   'EMA_10', 'EMA_50', 'RSI', 'MACD', 'MACD_signal',
                   'Upper_BB', 'Middle_BB', 'Lower_BB', 'HV', 'ATR']]
    target = df['close'].shift(-4)
    return features[:-4], target[:-4]

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def train_model(X_train, y_train):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(random_state=42))
    ])
    params = {
        'rf__n_estimators': [100],
        'rf__max_depth': [10],
        'rf__min_samples_split': [5]
    }
    grid = GridSearchCV(pipe, params, cv=3, scoring='neg_mean_squared_error')
    grid.fit(X_train, y_train)
    return grid.best_estimator_

# ---------- STREAMLIT APP ----------
st.title("🔮 BTC 4-Hour Price Predictor")

if st.button("🚀 Start Prediction"):
    with st.spinner("Fetching and processing data..."):
        df = fetch_data()
        df = compute_indicators(df)
        X, y = prepare_data(df)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
        model = train_model(X_train, y_train)

        # Evaluate
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        mae = np.mean(np.abs(y_test - y_pred))
        mape = mean_absolute_percentage_error(y_test, y_pred)

        st.success("✅ Model trained and evaluated!")
        st.write(f"**MSE:** {mse:.2f}")
        st.write(f"**MAE:** {mae:.2f}")
        st.write(f"**R² Score:** {r2:.4f}")
        st.write(f"**MAPE (Error Rate %):** {mape:.2f}%")

        # Predict next 4-hour BTC price
        current_features = X.iloc[[-1]]
        future_price = model.predict(current_features)[0]
        st.subheader(f"📈 Predicted BTC Price (next 4 hours): ${future_price:.2f}")

        # Plot predictions
        st.line_chart(pd.DataFrame({
            'Actual': y_test.values,
            'Predicted': y_pred
        }).reset_index(drop=True))



2025-04-19 18:40:11.564 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-04-19 18:40:13.067 
  command:

    streamlit run C:\Users\kamran.ahmad\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
